In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Thu Apr 23 10:27:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             44W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%capture
# For Qwen3-VL
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

# Configuration

In [4]:
SEED = 42

# Model configuration
MODEL_ID = 'alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K'
RESUME_MODEL_ID = None
RESUME_STEP = None

# Data configuration
SFT_DATA_ID = 'alxxtexxr/coco_captions_1.25k_serialized_for_Qwen3-VL-8B-Thinking-VCB8K'
SFT_COT_DATA_ID = 'alxxtexxr/R1-Onevision-1.25K'

# Training configuration
MINI_BATCH_SIZE = 4
GRAD_ACC_STEPS = 4
NUM_EPOCHS = 20
WARMUP_STEPS = 100
LR = 2e-4

# NOTE: 
# steps_per_epoch = train_size / batch_size = 2000 / 16 = 250 ~> 250
# warmup_steps    = steps_per_epoch * num_epochs_ideal * warmup_ratio = 250 * 10 * 0.05 = 125 ~> 100

In [5]:
from datetime import datetime

# Resume training configuration
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    model_name = RESUME_MODEL_ID
    hub_model_id = RESUME_MODEL_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist!"
            
else:
    # TODO: Handle the Hugging Face username properly
    model_name = MODEL_ID
    hub_model_id = f'{MODEL_ID}-SFT-QLoRA-v{datetime.now().strftime("%Y%m%d%H%M%S")}'
project_name = hub_model_id.split('/')[-1]

print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", model_name)
print("Project name:", project_name)
print("Hugging Face model ID:", hub_model_id)

Resume from checkpoint: False
Model name: alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K
Project name: Qwen3-VL-8B-Thinking-VCB8K-SFT-QLoRA-v20260423102806
Hugging Face model ID: alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-SFT-QLoRA-v20260423102806


In [6]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print("Set random seed:", seed)

set_seed(SEED)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Set random seed: 42


In [7]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [8]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = resume_from_checkpoint if isinstance(resume_from_checkpoint, str) else model_name,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # dtype = torch.float16, # Force FP16
)

==((====))==  Unsloth 2026.4.7: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
tokenizer.tokenizer.encode("<vthink><|vtok_7550|><|vtok_974|><|vtok_4167|><|vtok_4384|></vthink><think></think>")

[151669, 159221, 152645, 155838, 156055, 151670, 151667, 151668]

In [10]:
from peft import PeftModelForCausalLM

if not isinstance(model, PeftModelForCausalLM):
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers     = False, # False if not finetuning vision layers
        finetune_language_layers   = True,  # False if not finetuning language layers
        finetune_attention_modules = True,  # False if not finetuning attention layers
        finetune_mlp_modules       = True,  # False if not finetuning MLP layers

        r = 16,          # The larger, the higher the accuracy, but might overfit
        lora_alpha = 16, # Recommended alpha == r at least
        lora_dropout = 0,
        bias = 'none',
        random_state = SEED,
        use_rslora = False,  # We support rank stabilized LoRA
        loftq_config = None, # And LoftQ
        # target_modules = 'all-linear', # Optional now! Can specify a list if needed
    )
else:   
    print("Model is already a PeftModelForCausalLM, skipping PEFT wrapping.")

# Data

In [11]:
from datasets import load_dataset

dataset = load_dataset(SFT_DATA_ID)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
        num_rows: 1000
    })
    val: Dataset({
        features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
        num_rows: 125
    })
    test: Dataset({
        features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
        num_rows: 125
    })
})


In [12]:
train_set = dataset['train']
val_set = dataset['val']

print("Train set:")
print(train_set)
print("\nValidation set:")
print(val_set)

Train set:
Dataset({
    features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
    num_rows: 1000
})

Validation set:
Dataset({
    features: ['filename', 'cocoid', 'caption', 'image', 'image_serialized'],
    num_rows: 125
})


In [13]:
VTHINK_START = "<vthink>"
VTHINK_END = "</vthink>"

def convert_to_conversation(sample):
    conversation = [
        { 
            "role": "user",
            "content" : [
                {
                    "type" : "text",  
                    # "text": f"Description: {sample['caption']}\n\nBased on the description, provide the visual representations between {VTHINK_START} and {VTHINK_END}",
                    # "text": f"{sample['caption']}\n\nBased on the description above, provide the visual representations between {VTHINK_START} and {VTHINK_END}",
                    "text": f"Generate visual representations for this description between {VTHINK_START} and {VTHINK_END}.\n\nDescription: {sample['caption']}",
                },
                # {"type" : "image", "image" : sample["image"]},
            ],
        },
        { 
            "role" : "assistant",
            "content" : [
                {"type" : "text",  "text"  : f"<vthink>{sample['image_serialized']}</vthink>"},
            ],
        },
    ]
    return { "messages" : conversation }

train_dataset = [convert_to_conversation(sample) for sample in train_set]
val_dataset = [convert_to_conversation(sample) for sample in val_set]

print("Train dataset sample:")
print(train_dataset[0])
print("\nValidation dataset sample:")
print(val_dataset[0])

Train dataset sample:
{'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'Generate visual representations for this description between <vthink> and </vthink>.\n\nDescription: A man wearing a glove with a bird on top of it.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': '<vthink><|vtok_7550|><|vtok_974|><|vtok_4167|><|vtok_4384|><|vtok_4384|><|vtok_4384|><|vtok_4384|><|vtok_6079|><|vtok_4384|><|vtok_2710|><|vtok_2710|><|vtok_7983|><|vtok_2710|><|vtok_3179|><|vtok_2652|><|vtok_4280|><|vtok_6980|><|vtok_3407|><|vtok_6678|><|vtok_5357|><|vtok_3221|><|vtok_6079|><|vtok_4978|><|vtok_5639|><|vtok_3631|><|vtok_5062|><|vtok_6079|><|vtok_2710|><|vtok_7567|><|vtok_4143|><|vtok_1177|><|vtok_1384|><|vtok_5091|><|vtok_6152|><|vtok_4154|><|vtok_5949|><|vtok_4384|><|vtok_6637|><|vtok_3909|><|vtok_6079|><|vtok_6079|><|vtok_6079|><|vtok_7874|><|vtok_4154|><|vtok_1812|><|vtok_5357|><|vtok_6234|><|vtok_6555|><|vtok_7917|><|vtok_5683|><|vtok_7654|><|vtok_6079|><|vtok_4384|

# CoT Data

In [14]:
cot_dataset = load_dataset(SFT_COT_DATA_ID)
print(cot_dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'image', 'instruction', 'response'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['id', 'image', 'instruction', 'response'],
        num_rows: 125
    })
    test: Dataset({
        features: ['id', 'image', 'instruction', 'response'],
        num_rows: 125
    })
})


In [15]:
cot_train_set = cot_dataset['train']
cot_val_set = cot_dataset['validation']

print("CoT train set:")
print(cot_train_set)
print("\nCoT validation set:")
print(cot_val_set)

CoT train set:
Dataset({
    features: ['id', 'image', 'instruction', 'response'],
    num_rows: 1000
})

CoT validation set:
Dataset({
    features: ['id', 'image', 'instruction', 'response'],
    num_rows: 125
})


In [16]:
def convert_cot_sample_to_conversation(sample):
    conversation = [
        { 
            "role": "user",
            "content" : [
                {
                    "type" : "text",
                    "text": sample["instruction"],
                },
                {"type" : "image", "image" : sample["image"]},
            ],
        },
        { 
            "role" : "assistant",
            "content" : [
                {
                    "type" : "text",
                    "text"  : sample["instruction"],
                },
            ],
        },
    ]
    return { "messages" : conversation }

cot_train_dataset = [convert_cot_sample_to_conversation(sample) for sample in cot_train_set]
cot_val_dataset = [convert_cot_sample_to_conversation(sample) for sample in cot_val_set]

print("CoT train dataset sample:")
print(cot_train_dataset[0])
print("\nCoT validation dataset sample:")
print(cot_val_dataset[0])

CoT train dataset sample:
{'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'Please answer the question and provide the correct option letter, e.g., A, B, C, D, at the end. Provide your final answer between <answer> and </answer>.\n\nQuestion: Find m \\angle 1.\nChoices:\n(A) 31\n(B) 36\n(C) 67\n(D) 77'}, {'type': 'image', 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=403x188 at 0x7A49954D1FD0>}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': 'Please answer the question and provide the correct option letter, e.g., A, B, C, D, at the end. Provide your final answer between <answer> and </answer>.\n\nQuestion: Find m \\angle 1.\nChoices:\n(A) 31\n(B) 36\n(C) 67\n(D) 77'}]}]}

CoT validation dataset sample:
{'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'Please answer the question and provide the final answer at the end. Provide your final answer between <answer> and </answer>.\n\nQuestion: If each goose lays 3 eggs, how

In [17]:
# Combine the main dataset and CoT dataset, and tag them with 'is_cot' for later balanced sampling
from datasets import Dataset

def tag_cot(dataset, is_cot):
    for x in dataset:
        x['is_cot'] = is_cot
    return dataset

train_dataset = Dataset.from_list(tag_cot(train_dataset, False) + tag_cot(cot_train_dataset, True))
eval_dataset  = Dataset.from_list(tag_cot(val_dataset, False) + tag_cot(cot_val_dataset, True))

# Training

In [18]:
import random
from torch.utils.data import Sampler
from trl import SFTTrainer

class BalancedBatchSampler(Sampler):
    def __init__(self, dataset, batch_size):
        assert batch_size % 2 == 0
        self.dataset = dataset
        self.batch_size = batch_size
        self.half = batch_size // 2

        self.reg_indices = [
            i for i, x in enumerate(dataset) if not x['is_cot']
        ]
        self.cot_indices = [
            i for i, x in enumerate(dataset) if x['is_cot']
        ]

    def __iter__(self):
        random.shuffle(self.reg_indices)
        random.shuffle(self.cot_indices)

        reg_ptr, cot_ptr = 0, 0

        while reg_ptr + self.half <= len(self.reg_indices) and \
              cot_ptr + self.half <= len(self.cot_indices):

            batch = (
                self.reg_indices[reg_ptr:reg_ptr+self.half] +
                self.cot_indices[cot_ptr:cot_ptr+self.half]
            )

            random.shuffle(batch)
            yield batch

            reg_ptr += self.half
            cot_ptr += self.half

    def __len__(self):
        return min(len(self.reg_indices), len(self.cot_indices)) // self.half

class BalancedSFTTrainer(SFTTrainer):
    def get_train_dataloader(self):
        sampler = BalancedBatchSampler(
            self.train_dataset,
            self.args.per_device_train_batch_size
        )

        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_sampler=sampler,
            collate_fn=self.data_collator,
        )

In [19]:
import math
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTConfig
from transformers import EarlyStoppingCallback

FastVisionModel.for_training(model) # Enable for training

max_steps = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACC_STEPS)) * NUM_EPOCHS
print("Calculated max steps:", max_steps)

trainer = BalancedSFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        # Training arguments
        seed = SEED,
        
        per_device_train_batch_size = MINI_BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACC_STEPS,
        
        # max_steps = 50, # For testing
        max_steps = max_steps,
        # num_train_epochs = NUM_EPOCHS,
        # warmup_ratio=0.05,
        warmup_steps = WARMUP_STEPS,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        max_grad_norm=1.0,
        weight_decay = 0.01,
        
        # Validation arguments
        eval_strategy='steps',
        eval_steps=20,
        
        # Logging arguments
        logging_strategy='steps',
        logging_steps=10,
        # logging_first_step=True,
        report_to=['tensorboard', 'wandb'],
        
        # Saving arguments
        save_strategy='steps',
        # save_steps=1, # For testing
        save_steps=20,
        # save_total_limit=5, # 1 best + 4 recent checkpoints. Warning: It doesn't work
        
        # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
        # So you will find one checkpoint at the end of each epoch.
        # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
        load_best_model_at_end=True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,

        output_dir=project_name,
        hub_model_id=hub_model_id,
        push_to_hub=True,
        hub_strategy='all_checkpoints',
        hub_always_push=True,

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 1024,
    ),
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience = 2,
            early_stopping_threshold = 0.001,
        )
    ]
)

Calculated max steps: 2500
Unsloth: Model does not have a default image size - using 512


In [20]:
trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 20 | Total steps = 2,500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 43,646,976 of 8,875,708,656 (0.49% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
20,7.218400,5.506761


KeyboardInterrupt: 

# Inference

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference

image = None
instruction = val_dataset[0]['messages'][0]['content'][0]['text']

messages = [
    {"role": "user", "content": [
        # {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

<|im_start|>user
A baseball player with one leg kicked up preparing to throw a ball

Based on the description above, provide the visual representations between <vthink> and </vthink><|im_end|>
<|im_start|>assistant
<think>
</think>

<vthink><|vtok_409|><|vtok_1205|><|vtok_4850|><|vtok_3632|><|vtok_3188|><|vtok_4850|><|vtok_1856|><|vtok_7883|><|vtok_1856|><|vtok_874|><|vtok_4850|><|vtok_3632|><|vtok_874|><|vtok_2609|><|vtok_3632|><|vtok_7550|><|vtok_2379|><|vtok_3632|><|vtok_3632|><|vtok_3188|><|vtok_2609|><|vtok_7883|><|vtok_7883|><|vtok_874|><|vtok_874|><|vtok_7714|><|vtok_2951|><|vtok_1075|><|vtok_874|><|vtok_3632|><|vtok_874|><|vtok_3992|><|vtok_2379|><|vtok_3634|><|vtok_3632|><|vtok_3632|><|vtok_874|><|vtok_3634|><|vtok_3632|><|vtok_7714|><|vtok_3188|><|vtok_874|><|vtok_874|><|vtok_1856|><|vtok_7714|><|vtok_1075|><|vtok_1145|><|vtok_1145|><|vtok_1744|><|vtok_3634|><|vtok_3188|><|vtok_3640|><|vtok_3640|><|vtok_665|><|vtok_636|><|vtok_6047|><|vtok_3188|><|vtok_3632|><|vtok_1075|><|vt